# 02 · Spectral analysis

Most impairments and most receiver algorithms are easiest to understand in
the frequency domain.  This notebook covers the spectral tools the later
notebooks use:

1. the **power spectral density** (PSD) and Welch's estimator: resolution
   against variance;
2. reading absolute levels off a PSD: noise density and total power;
3. moving a spectrum: `shift_frequency` and the FFT-bin grid;
4. **pilot tones**, the narrow lines several estimators lock onto;
5. the **spectrogram** for spectra that change over time.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    """Device->host transfer for hand-made Matplotlib plots."""
    return to_device(a, "cpu")

## 1. Welch's estimator: resolution against variance

The PSD `S(f)` is the power per hertz at frequency `f`.  A single FFT of
the whole record (the periodogram) has the finest resolution, but its
values scatter as wildly as the noise itself.  Welch's method cuts the
record into segments of `nperseg` samples, windows them, and averages their
periodograms:

* the frequency resolution is about `fs / nperseg` (times the window's
  width factor);
* averaging `K` segments shrinks the scatter by about `sqrt(K)`.

Below, a weak tone 30 dB under a white-noise floor in total power:
short segments give a smooth floor but smear the tone; long segments
resolve it at the price of a noisy floor.

In [ ]:
fs = 1e6
n = 2**18
rng = np.random.default_rng(0)
noise = (rng.standard_normal(n) + 1j * rng.standard_normal(n)) / np.sqrt(
    2
)  # unit power
tone = 10 ** (-30 / 20) * np.exp(2j * np.pi * 123.4e3 * np.arange(n) / fs)
x = ck.backend.to_device((noise + tone).astype(np.complex64), DEVICE)

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True, sharey=True)
for ax, nperseg in zip(axes, (256, 4096, 65536)):
    f, p = ck.spectral.welch_psd(x, sampling_rate=fs, nperseg=nperseg)
    order = np.argsort(host(f))
    ax.plot(host(f)[order] / 1e3, 10 * np.log10(host(p)[order]))
    ax.set_title(
        f"nperseg = {nperseg}: {fs / nperseg:.0f} Hz bins, K = {2 * n // nperseg - 1} segments"
    )
    ax.set_ylabel("PSD [dB/Hz]")
axes[-1].set_xlim(100, 150)
axes[-1].set_xlabel("Frequency [kHz]")
plt.show()

## 2. Reading levels off a PSD

With density scaling, a white noise of power `σ²` spread over the
two-sided band `fs` has the flat level `σ²/fs`, and integrating the PSD
over frequency returns the total power (Parseval).  Both checks matter
later: the noise floor gives the SNR, and the integral tells whether a
block changed the power.

In [ ]:
f, p = ck.spectral.welch_psd(x, sampling_rate=fs, nperseg=4096)
f, p = host(f), host(p)
df = fs / 4096
print(
    f"median floor  = {10 * np.log10(np.median(p)):.2f} dB/Hz  (10 log10(1/fs) = {10 * np.log10(1 / fs):.2f})"
)
print(
    f"integrated    = {np.sum(p) * df:.4f}  (sample power {float(np.mean(np.abs(host(x)) ** 2)):.4f})"
)

## 3. The spectrum of a modulated signal

A 16-QAM signal at 10 GBd with a root-raised-cosine pulse of roll-off
`β = 0.2` occupies `(1 + β) · Rs = 12 GHz`, flat in the middle with
cosine-shaped edges.  `plot_psd` takes the `Signal` directly (the sampling
rate comes with it); the pulse itself is the subject of
[04 · Pulse shaping](04-pulse-shaping-and-matched-filtering.ipynb).

In [ ]:
tx = ck.generate(
    Constellation.qam(16),
    num_symbols=2**15,
    symbol_rate=10e9,
    sps=4,
    pulse=RRC(rolloff=0.2),
    rng=1,
    device=DEVICE,
)
fig, ax = plt.subplots(figsize=(8, 4))
ck.plotting.plot_psd(
    tx,
    nperseg=2048,
    ax=ax,
    title="16-QAM, 10 GBd, RRC β = 0.2 (dotted: ±(1 + β) Rs / 2)",
)
for edge in (-6e9, 6e9):
    ax.axvline(edge, color="gray", ls=":")
plt.show()

## 4. Shifting a spectrum

Multiplying by `exp(j2π f0 t)` moves the whole spectrum by `f0`; a carrier
frequency offset between two lasers does exactly this.  `shift_frequency`
snaps `f0` to the nearest FFT bin of the record (`fs / N`), so the shift is
a whole number of cycles and the record stays periodic.  `grid_frequency`
returns the value that is actually applied, which is the one to tell a
receiver.

In [ ]:
f0 = 2.0123e9
shifted = ck.spectral.shift_frequency(tx, frequency=f0)
applied = ck.spectral.grid_frequency(
    f0, sampling_rate=tx.sampling_rate, num_samples=tx.samples.shape[-1]
)
print(
    f"requested {f0 / 1e9:.6f} GHz, applied {applied / 1e9:.6f} GHz "
    f"(bin spacing {tx.sampling_rate / tx.samples.shape[-1] / 1e6:.3f} MHz)"
)

fig, ax = plt.subplots(figsize=(8, 4))
ck.plotting.plot_psd(tx, nperseg=2048, ax=ax, title=None, label="original")
ck.plotting.plot_psd(
    shifted, nperseg=2048, ax=ax, title="Frequency shift by 2 GHz", label="shifted"
)
ax.legend()
plt.show()

## 5. Pilot tones

A continuous-wave **pilot tone** is a narrow spectral line added next to
(or inside) the signal band.  It costs a little power and gives the
receiver a clean reference for the carrier frequency and phase
([10](10-frequency-offset-estimation.ipynb), [12](12-pilot-aided-carrier-recovery.ipynb)) and
for the polarization state ([16](16-polarization-multiplexing.ipynb)).  `power_ratio_db` sets
the tone power relative to the signal.

In [ ]:
piloted = ck.spectral.add_pilot_tone(tx, frequency=7e9, power_ratio_db=-15)
ck.plotting.plot_psd(
    piloted, nperseg=2048, title="Pilot tone at 7 GHz, -15 dB", show=True
)

## 6. The spectrogram

A PSD averages over the whole record.  When the spectrum moves (a drifting
laser, a frequency hop, a burst) the **spectrogram** shows the PSD of
consecutive short segments as an image: time on one axis, frequency on the
other.  Its resolution trade-off is the same as Welch's, now between time
and frequency.  Here a tone drifts linearly by 200 kHz over 10 ms, next to
a burst that starts halfway through.

In [ ]:
fs = 1e6
t = np.arange(10_000) / fs
chirp = np.exp(2j * np.pi * (-100e3 * t + 0.5 * 200e3 / t[-1] * t**2))
burst = (t > t[-1] / 2) * np.exp(2j * np.pi * 300e3 * t)
drift = ck.backend.to_device((chirp + burst).astype(np.complex64), DEVICE)
ck.plotting.plot_spectrogram(
    drift, sampling_rate=fs, nperseg=256, title="Drifting tone and a burst", show=True
)

**Summary**

* Welch PSD: `nperseg` sets resolution (`fs/nperseg`); the number of
  segments sets the variance.
* Density scaling: white noise of power `σ²` sits at `σ²/fs`, and the
  integral of the PSD is the total power.
* `shift_frequency` and `add_pilot_tone` snap to the FFT grid; ask
  `grid_frequency` for the applied value.

**Next:** [03 · Digital filters](03-digital-filters.ipynb).